# 05c - Penumpukan (Stacking) dan Voting
Materi Modul 5: penumpukan/stacking, lab klasifikasi ansambel dengan Python.

- **Voting**: gabungkan prediksi beberapa model (hard = suara terbanyak, soft = rata-rata probabilitas)
- **Stacking**: prediksi model-model dasar dijadikan **fitur** untuk sebuah *meta-model* (final estimator)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier

df = pd.read_csv('../datasets/breast_cancer.csv')
X, y = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

dasar = [
    ('lr', make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
    ('knn', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))),
    ('svm', make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
]

## 1. Model dasar masing-masing

In [ ]:
for nama, m in dasar:
    print(f'{nama:4s} | CV = {cross_val_score(m, X_train, y_train, cv=5).mean():.4f}')

## 2. Voting (hard dan soft)

In [ ]:
hard = VotingClassifier(dasar, voting='hard')
soft = VotingClassifier(dasar, voting='soft')

print('Voting hard | CV =', round(cross_val_score(hard, X_train, y_train, cv=5).mean(), 4))
print('Voting soft | CV =', round(cross_val_score(soft, X_train, y_train, cv=5).mean(), 4))

## 3. Stacking

In [ ]:
stack = StackingClassifier(estimators=dasar, final_estimator=LogisticRegression(max_iter=2000), cv=5)
stack.fit(X_train, y_train)

print('Stacking | CV           =', round(cross_val_score(stack, X_train, y_train, cv=5).mean(), 4))
print('Stacking | akurasi test =', round(stack.score(X_test, y_test), 4))

## 4. Apa yang dilihat meta-model?
Probabilitas dari tiap model dasar menjadi fitur untuk meta-model.

In [ ]:
meta_fitur = stack.transform(X_test)
print('Bentuk fitur meta:', meta_fitur.shape, '(1 kolom per model dasar)')
print(pd.DataFrame(meta_fitur, columns=[n for n, _ in dasar]).head().round(3))

print()
print('Bobot meta-model (regresi logistik):')
print(pd.Series(stack.final_estimator_.coef_[0], index=[n for n, _ in dasar]).round(2))

## 5. Rekap perbandingan (akurasi test)

In [ ]:
hasil = {}
for nama, m in dasar:
    hasil[nama] = m.fit(X_train, y_train).score(X_test, y_test)
hasil['voting soft'] = soft.fit(X_train, y_train).score(X_test, y_test)
hasil['stacking'] = stack.score(X_test, y_test)
pd.Series(hasil).sort_values(ascending=False).round(4)

## Catatan
- (beda voting dan stacking, kenapa stacking memakai cross-validation di dalamnya, kapan ansambel tidak membantu, dll.)